# APEX AI — Phase 6: Autonomous Technical Tool-Maker Fine-Tuning (QLoRA on Free Colab GPU)

This notebook demonstrates how to fine-tune **Llama-3-8B-Instruct** using **QLoRA (4-bit Low-Rank Adaptation)** on Google Colab's free T4 GPU to transform it into a **Senior F1 Technical Director & Autonomous Tool Maker**.

### What this accomplishes:
1. **Technical F1 Domain Specialization**: Forces the model to think and speak in terms of aerodynamic balance, tyre degradation slopes, lateral load dynamics, and telemetry deltas.
2. **Dynamic Tool Generation (LATM)**: When presented with a complex, novel query, the model writes custom Python/SQL code to query `f1.db` on-the-fly, executes it, and synthesizes the mathematical results.

---

## 1. Install Unsloth / Hugging Face QLoRA Libraries
(Unsloth accelerates fine-tuning 2-5x and reduces VRAM usage to fit easily in Colab's free 16GB T4 GPU).

In [16]:
%%capture
# 1. Cleanly update datasets first to resolve circular imports
!pip install -U --no-cache-dir -q datasets

# 2. Install pre-compiled compatible wheel packages of unsloth & xformers to bypass local compilation failures
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q --no-deps xformers trl peft accelerate bitsandbytes triton

## 2. Load 4-bit Quantized Llama-3-8B with Fast LoRA

In [21]:
# 1. Unregister pyarrow extension types if they already exist to prevent registration conflicts on reload
import pyarrow as pa
for ext_name in [
    "datasets.features.features.Array2DExtensionType",
    "datasets.features.features.Array3DExtensionType",
    "datasets.features.features.Array4DExtensionType",
    "datasets.features.features.Array5DExtensionType"
]:
    try:
        pa.unregister_extension_type(ext_name)
    except KeyError:
        pass

# 2. Forcefully purge cached datasets modules to resolve python memory's circular import loop
import sys
for module in list(sys.modules.keys()):
    if module == 'datasets' or module.startswith('datasets.'):
        sys.modules.pop(module, None)

import datasets
import torch
from unsloth import FastLanguageModel

max_seq_length = 2048
dtype = None # Auto detection (Float16 on T4)
load_in_4bit = True

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/llama-3-8b-Instruct-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

# Add LoRA adapters to attention and MLP layers
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 42,
)

/usr/local/lib/python3.13/dist-packages/datasets/filesystems/__init__.py:24: UserWarning: A filesystem protocol was already set for bz2 and will be overwritten.
  warnings.warn(f"A filesystem protocol was already set for {fs_class.protocol} and will be overwritten.")
/usr/local/lib/python3.13/dist-packages/datasets/filesystems/__init__.py:24: UserWarning: A filesystem protocol was already set for gzip and will be overwritten.
  warnings.warn(f"A filesystem protocol was already set for {fs_class.protocol} and will be overwritten.")
/usr/local/lib/python3.13/dist-packages/datasets/filesystems/__init__.py:24: UserWarning: A filesystem protocol was already set for lz4 and will be overwritten.
  warnings.warn(f"A filesystem protocol was already set for {fs_class.protocol} and will be overwritten.")
/usr/local/lib/python3.13/dist-packages/datasets/filesystems/__init__.py:24: UserWarning: A filesystem protocol was already set for xz and will be overwritten.
  warnings.warn(f"A filesystem prot

Unsloth: xformers is installed but cannot load with torch 2.14.1+cu130 (it was built for 2.10.0+cu128), so PyTorch attention is used instead.
To install a matching xformers, run `unsloth install-kernels xformers`, or `pip uninstall xformers` to hide this. UNSLOTH_ENABLE_LOGGING=1 shows the full error.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.11.3: Fast Llama patching. Transformers: 5.17.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.14.1+cu130. CUDA: 7.5. CUDA Toolkit: 13.0. Triton: 3.8.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Unsloth: Will load unsloth/llama-3-8b-Instruct-bnb-4bit as a legacy tokenizer.
Unsloth 2026.11.3 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


Unsloth: compiling each transformer block as one region (LlamaDecoderLayer).


## 3. Clone Repository & Load Technical Instruction Dataset

In [27]:
import os
from datasets import load_dataset

# Clone and checkout the master branch specifically
if not os.path.exists("repo"):
    !git clone -b master https://github.com/bhxvxsh/yeah-def-my-nyc-project.git repo
else:
    print("Repository already exists. Fetching and switching to master...")
    !cd repo && git fetch origin && git checkout master && git pull origin master

# Recursively locate the f1_technical_alpaca.json file
target_file = None
for root, dirs, files in os.walk("repo"):
    if "f1_technical_alpaca.json" in files:
        target_file = os.path.join(root, "f1_technical_alpaca.json")
        break

if target_file:
    print(f"Found target dataset file at: {target_file}")
else:
    # Find any JSON file in the repository as a fallback
    print("Target JSON not found by name. Searching for any JSON files in the repo:")
    for root, dirs, files in os.walk("repo"):
        for file in files:
            if file.endswith(".json") and ".git" not in root:
                target_file = os.path.join(root, file)
                print(f"Fallback option found: {target_file}")
                break
        if target_file:
            break

if not target_file:
    raise FileNotFoundError("Could not find any JSON dataset in the repository.")

print(f"Loading dataset from: {target_file}")
dataset = load_dataset("json", data_files=target_file, split="train")

prompt_template = """Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:
{}

### Response:
{}"""

def formatting_prompts_func(examples):
    instructions = examples["instruction"]
    outputs      = examples["output"]
    texts = []
    for instruction, output in zip(instructions, outputs):
        text = prompt_template.format(instruction, output) + tokenizer.eos_token
        texts.append(text)
    return { "text" : texts, }

dataset = dataset.map(formatting_prompts_func, batched = True)

Repository already exists. Fetching and switching to master...
branch 'master' set up to track 'origin/master'.
Switched to a new branch 'master'
From https://github.com/bhxvxsh/yeah-def-my-nyc-project
 * branch            master     -> FETCH_HEAD
Already up to date.
Found target dataset file at: repo/03_Training_Dataset/f1_technical_alpaca.json
Loading dataset from: repo/03_Training_Dataset/f1_technical_alpaca.json


Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/3 [00:00<?, ? examples/s]

## 4. Train the Model (Takes ~15-20 minutes on free T4 GPU)

In [28]:
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 60,
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 42,
        output_dir = "outputs",
    ),
)

trainer_stats = trainer.train()

Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/3 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 3 | Num Epochs = 60 | Total steps = 60
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 41,943,040 of 8,072,204,288 (0.52% trained)


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: packed / padding-free rows run each transformer block uncompiled; only unpacked batches use the compiled blocks.
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss
1,2.701989
2,2.702089
3,2.661848
4,2.522015
5,2.328901
6,2.154326
7,1.947718
8,1.695436
9,1.428648
10,1.172097


Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-60/tokenizer_config.json.


## 5. Test Autonomous Tool Generation on a Complex Query

In [29]:
FastLanguageModel.for_inference(model)

test_prompt = """Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:
Evaluate the net position delta and mechanical recovery efficiency of Sergio Perez during the 2023 season when starting outside the top 5.

### Response:
"""

inputs = tokenizer([test_prompt], return_tensors = "pt").to("cuda")
outputs = model.generate(**inputs, max_new_tokens = 512, use_cache = True)
print(tokenizer.batch_decode(outputs)[0])

Both `max_new_tokens` (=512) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


<|begin_of_text|>Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:
Evaluate the net position delta and mechanical recovery efficiency of Sergio Perez during the 2023 season when starting outside the top 5.

### Response:
### Technical Telemetry & Race Craft Evaluation: S. Perez (2023)

To isolate recovery efficiency from sub-optimal qualifying positions, we analyze grid-to-flag deltas conditioned on $P_{\text{grid}} > 5$.

```python
query = '''
    SELECT round, event_name, GridPosition, Position, (GridPosition - Position) as net_delta, Laps
    FROM race_results
    WHERE DriverId = 'perez' AND year = 2023 AND GridPosition > 5 AND finished = 1
    ORDER BY net_delta DESC
'''
df = pd.read_sql(query, con)
mean_recovery = df['net_delta'].mean()
std_recovery = df['net_delta'].std()
result = {
    "sample_size": len(df),
    "mean_positions_gained": round(mean_recovery, 2),
    "std_dev": round(std_recovery, 2),
    

In [32]:
# ==============================================================================
# APEX AI — Robust Colab LLM Server (Unsloth / Llama-3-8B)
# ==============================================================================
import nest_asyncio
from fastapi import FastAPI
from pydantic import BaseModel
import uvicorn
from pyngrok import ngrok, exception
import torch
import traceback
import threading

# 1. Safely load or verify model & tokenizer
try:
    from unsloth import FastLanguageModel

    # If model is not defined in the namespace, let's load it.
    if 'model' not in globals() or 'tokenizer' not in globals():
        print("Model/Tokenizer not found in memory. Loading from pre-trained unsloth model...")
        max_seq_length = 2048
        dtype = None
        load_in_4bit = True
        model, tokenizer = FastLanguageModel.from_pretrained(
            model_name = "unsloth/llama-3-8b-Instruct-bnb-4bit",
            max_seq_length = max_seq_length,
            dtype = dtype,
            load_in_4bit = load_in_4bit,
        )

    FastLanguageModel.for_inference(model)
    model.eval()
except Exception as e:
    print(f"Warning during model preparation: {e}")

app = FastAPI()

class QueryRequest(BaseModel):
    query: str

@app.get("/health")
def health():
    return {"status": "ok", "model": "Llama-3-8B-F1-Technical"}

@app.post("/ask")
def ask(req: QueryRequest):
    try:
        user_text = req.query.strip()
        print(f"\n[Incoming User Query]: {user_text}")

        prompt = (
            "Below is an instruction that describes a task. Write a response that appropriately completes the request.\n\n"
            f"### Instruction:\n{user_text}\n\n"
            "### Response:\n"
        )

        inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

        with torch.inference_mode():
            outputs = model.generate(
                input_ids=inputs.input_ids,
                attention_mask=inputs.attention_mask,
                max_new_tokens=350,
                temperature=0.7,
                top_p=0.9,
                use_cache=True,
                pad_token_id=tokenizer.eos_token_id,
            )

        # Decode only the generated response
        generated_tokens = outputs[0][inputs.input_ids.shape[1]: ]
        answer = tokenizer.decode(generated_tokens, skip_special_tokens=True).strip()

        print(f"[Generated Response]: {answer[:100]}...")
        return {"response": answer}

    except Exception as e:
        err_msg = f"{type(e).__name__}: {str(e)}\n{traceback.format_exc()}"
        print(f"[ERROR in /ask]: {err_msg}")
        return {"response": f"Colab Execution Error: {str(e)}"}

# 2. Start Ngrok Tunnel securely
try:
    ngrok.set_auth_token("3LbYUgS8NEuIbPBhgOda2g3osfJ_2FdXmg6WMDRecwSUgEi1Q")
except exception.PyngrokNgrokError:
    pass

# Clean existing tunnels to avoid collisions
ngrok.kill()
public_url = ngrok.connect(8000).public_url

print("\n" + "=" * 65)
print(f"🚀 LIVE APEX AI COLAB URL: {public_url}")
print("=" * 65 + "\n")

nest_asyncio.apply()

# Run server in background thread
def run_server():
    uvicorn.run(app, host="0.0.0.0", port=8000, log_level="info")

threading.Thread(target=run_server, daemon=True).start()


🚀 LIVE APEX AI COLAB URL: https://fasting-washout-crazy.ngrok-free.dev



INFO:     Started server process [3541]
